In [ ]:
#Air resistance
import numpy as np
from scipy.special import ellipk

# -----------------------------
# Constants
# -----------------------------
g = 9.81
rho_air = 1.225        # kg/m^3
C_d = 1.1              # cylinder, transverse flow

# -----------------------------
# Core physics
# -----------------------------
def physical_pendulum_period(L, alpha, theta0,
                             crosssection,
                             m_total):
    """
    Physical pendulum with:
    - exact angle dependence
    - cylindrical bob (Square crosssection)
    - weak quadratic air resistance

    Parameters
    ----------
    L : float
        Length (m)
    alpha : float
        Rod mass fraction
    theta0 : float
        Initial angle (rad)
    crosssection : float
        Square crosssection (m**2)
    m_total : float
        Total mass (kg)

    Returns
    -------
    T : float
        Period (s)
    """

    # -----------------------------
    # Small-angle physical pendulum
    # -----------------------------
    shape = ((1/3)*alpha + (1 - alpha)) / ((1/2)*alpha + (1 - alpha))
    T0 = 2 * np.pi * np.sqrt(L / g * shape)

    # -----------------------------
    # Exact finite-angle correction
    # -----------------------------
    k2 = np.sin(theta0 / 2)**2
    angle_factor = (2 / np.pi) * ellipk(k2)

    # -----------------------------
    # Air resistance correction
    # -----------------------------
    A = crosssection
    omega0 = 2 * np.pi / T0

    gamma = (rho_air * C_d * A * L) / m_total
    drag_factor = 1 + (gamma / omega0)**2 / 8

    return T0 * angle_factor * drag_factor

# -----------------------------
# Measurement noise
# -----------------------------
def add_measurement_noise(L, T,
                          sigma_L=0.002,
                          sigma_T=0.015):
    return (
        L + np.random.normal(0, sigma_L),
        T + np.random.normal(0, sigma_T)
    )

# -----------------------------
# Dataset generator
# -----------------------------
def generate_dataset_AR(N, L_range=(0.3, 1.2),
                        alpha_range=(0.1, 0.3),
                        theta_range=(0.05, 0.6),
                        crosssection_range=(0.01, 0.04),
                        m_total_range=(0.1, 1.0)):
    X, y = [], []

    for _ in range(N):
        L = np.random.uniform(*L_range)
        alpha = np.random.uniform(*alpha_range)
        theta0 = np.random.uniform(*theta_range)
        crosssection = np.random.uniform(*crosssection_range)   # 1–4 cm^2 crosssection
        m_total = np.random.uniform(*m_total_range)  # 0.1-1.0 kg

        T_true = physical_pendulum_period(
            L, alpha, theta0, crosssection, m_total
        )

        Lm, Tm = add_measurement_noise(L, T_true)

        X.append([Lm, alpha, theta0, crosssection, m_total])
        y.append(Tm)

    return np.array(X), np.array(y)

# -----------------------------
# Example
# -----------------------------
if __name__ == "__main__":
    np.random.seed(0)
    X, y = generate_dataset_AR(5000)

    print("Features: [L, alpha, theta0, crosssection, m_total]")
    print(X[:5])
    print(y[:5])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Generate dataset
np.random.seed(42)
X, y = generate_dataset_AR(100000,                   #Antal samples 
                        L_range=(0.01, 3.0),       #længde  [m]
                        alpha_range=(0.0, 1.0),   #mass fraction 
                        theta_range=(0.05,2.5),    #initial angle [rad]
                        crosssection_range=(0.01, 0.5),    #crosssection [m^2]
                        m_total_range=(0.01, 5.0)) #total vægt [kg]
# Extract length and period
L_measured = X[:, 0]
T_measured = y

# Create theoretical curve for simple pendulum
L_theory = np.linspace(min(L_measured), max(L_measured), len(L_measured//10))
T_theory = 2 * np.pi * np.sqrt(L_theory / g)

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(L_measured, T_measured, alpha=0.3, s=10, label='Simulated measurements')
plt.plot(L_theory, T_theory, 'r-', linewidth=2, label='Simple pendulum theory: $T = 2\pi\sqrt{L/g}$')
plt.xlabel('Length L (m)', fontsize=12)
plt.ylabel('Period T (s)', fontsize=12)
plt.title('Pendulum Period vs Length', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
#Save the above generated dataset to csv
import numpy as np
import pandas as pd
# Generate dataset
np.random.seed(42)
X, y = generate_dataset_AR(200000,                   #Antal samples 
                        L_range=(0.01, 3.0),       #længde  [m]
                        alpha_range=(0.0, 1.0),   #mass fraction 
                        theta_range=(0.05,2.5),    #initial angle [rad]
                        crosssection_range=(0.01, 0.5),    #crosssection [m^2]
                        m_total_range=(0.01, 5.0)) #total vægt [kg]
# Create a DataFrame
df = pd.DataFrame(X, columns=['L_measured', 'alpha', 'theta0', 'crosssection', 'm_total'])
df['Period'] = y
# Save to CSV
df.to_csv('pendulum_data_EXTREME_FULL.csv', index=False)